In [ ]:

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns




In [ ]:

books = pd.read_csv(
    '../data/raw/Books.csv',
    low_memory=False
)

ratings = pd.read_csv(
    '../data/raw/Ratings.csv'
)

users = pd.read_csv(
    '../data/raw/Users.csv'
)


In [ ]:

print("Books:", books.shape)
print("Ratings:", ratings.shape)
print("Users:", users.shape)


In [ ]:

books.head()


In [ ]:

users.head()


In [ ]:

ratings.head()


In [ ]:

books.info()
ratings.info()
users.info()


In [ ]:

print("BOOKS")
print(books.isnull().sum())

print("\nRATINGS")
print(ratings.isnull().sum())

print("\nUSERS")
print(users.isnull().sum())


In [ ]:

ratings['Book-Rating'].value_counts().sort_index()


In [ ]:

print("Duplicate books:", books.duplicated().sum())
print("Duplicate ISBNs:", books['ISBN'].duplicated().sum())

print("Duplicate ratings:", ratings.duplicated().sum())
print("Duplicate user-book pairs:",
      ratings.duplicated(subset=['User-ID', 'ISBN']).sum())

print("Duplicate users:", users['User-ID'].duplicated().sum())


In [ ]:

print(books['Year-Of-Publication'].value_counts().head(15))

print(users['Age'].describe())

print(ratings['Book-Rating'].describe())


In [ ]:

books = books.rename(columns={
    'Book-Title': 'title',
    'Book-Author': 'author',
    'Year-Of-Publication': 'year',
    'Publisher': 'publisher',
    'Image-URL-S': 'image_s',
    'Image-URL-M': 'image_m',
    'Image-URL-L': 'image_l',
    'ISBN': 'isbn'
})

ratings = ratings.rename(columns={
    'User-ID': 'user_id',
    'ISBN': 'isbn',
    'Book-Rating': 'rating'
})

users = users.rename(columns={
    'User-ID': 'user_id',
    'Location': 'location',
    'Age': 'age'
})


In [ ]:

books['author'] = books['author'].fillna('Unknown')
books['publisher'] = books['publisher'].fillna('Unknown')

books['image_l'] = books['image_l'].fillna(
    books['image_m']
)


In [ ]:

books.isnull().sum()


In [ ]:

books['year'] = pd.to_numeric(
    books['year'],
    errors='coerce'
)

books['year'].isnull().sum()


In [ ]:

books[
    (books['year'] < 1450) |
    (books['year'] > 2026) |
    (books['year'].isna())
][['isbn', 'title', 'year']].head(20)


In [ ]:

books.loc[
    (books['year'] < 1450) |
    (books['year'] > 2026),
    'year'
] = np.nan


In [ ]:

users.loc[
    (users['age'] < 5) |
    (users['age'] > 100),
    'age'
] = np.nan


In [ ]:

users['age'].describe()


In [ ]:

explicit_ratings = ratings[
    ratings['rating'] > 0
].copy()

implicit_ratings = ratings[
    ratings['rating'] == 0
].copy()

print("Explicit ratings:", explicit_ratings.shape)
print("Implicit interactions:", implicit_ratings.shape)


In [ ]:

missing_isbns = ~ratings['isbn'].isin(books['isbn'])

print("Ratings with unknown ISBNs:",
      missing_isbns.sum())


In [ ]:

ratings = ratings[
    ratings['isbn'].isin(books['isbn'])
].copy()

explicit_ratings = ratings[
    ratings['rating'] > 0
].copy()

implicit_ratings = ratings[
    ratings['rating'] == 0
].copy()


In [ ]:

ratings_with_books = explicit_ratings.merge(
    books,
    on='isbn',
    how='inner',
    validate='many_to_one'
)

ratings_with_books.head()


In [ ]:

print(ratings_with_books.shape)

print(ratings_with_books[
    ['user_id', 'isbn', 'title', 'author', 'rating']
].head())


In [ ]:

from pathlib import Path

output_dir = Path('../data/processed')
output_dir.mkdir(parents=True, exist_ok=True)

books.to_csv(output_dir / 'books_clean.csv', index=False)
users.to_csv(output_dir / 'users_clean.csv', index=False)
ratings.to_csv(output_dir / 'ratings_clean.csv', index=False)

explicit_ratings.to_csv(
    output_dir / 'explicit_ratings.csv',
    index=False
)

ratings_with_books.to_csv(
    output_dir / 'ratings_with_books.csv',
    index=False
)

print("Cleaned datasets saved successfully!")
